# autoresearch-team — Kaggle run (GPU T4 x2)

Save & Run All. Before starting: add the secret `GEMINI_API_KEY`
(Add-ons → Secrets) and attach this project as a dataset (or clone it in
cell 1). Everything persists under `/kaggle/working`; reattach that output
as an input dataset in the next session to resume (state.json, E11).

In [ ]:
# 1. Get the project (clone is the default path; dataset attach also works),
# install deps, and make sure the nested training repo has its own .git.
import os, subprocess, sys
from pathlib import Path
PROJECT = Path('/kaggle/working/autosearch_kaggle')
subprocess.run(['git', 'clone', '-q',
                'https://github.com/MokshonWork/autosearch_kaggle.git',
                str(PROJECT)], check=True)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
# if the upload stripped repo/.git, re-init it (orchestrator needs worktrees):
if not (PROJECT / 'repo' / '.git').exists():
    for a in (['init', '-q'], ['add', '-A'],
              ['-c', 'user.email=k@kaggle', '-c', 'user.name=k',
               'commit', '-q', '-m', 't4-port baseline']):
        subprocess.run(['git', '-C', str(PROJECT / 'repo'), *a], check=True)
    print('repo/.git re-initialized')
# torch/cuda is PREINSTALLED on the Kaggle image (verified 2.11.0+cu128);
# do not pin/downgrade it — only add what the harness needs.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'rustbpe>=0.1.0', 'tiktoken>=0.11.0'], check=True)
print(subprocess.run(['nvidia-smi', '--list-gpus'],
                     capture_output=True, text=True).stdout)
import torch
print('torch', torch.__version__, 'cuda', torch.version.cuda,
      'devices', torch.cuda.device_count())

In [ ]:
# 2. Secrets -> env (A4). Key only in this process's env; the training
# subprocess gets a whitelisted env without it (G7, E10). Never printed.
#
# NOTE: we do NOT use kaggle_secrets.UserSecretsClient. Kaggle injects
# KAGGLE_URL_BASE=https://api.kaggle.com, but the UserSecretsService endpoint
# is served only by https://www.kaggle.com; against the injected host every
# request gets HTTP 400 with an empty body, which kaggle_web_client flattens
# into a generic ConnectionError ("flaky backend" red herring).
# src/kaggle_secret.py hits the working host and separates "not attached to
# this kernel" from real backend failures.
import os, sys
sys.path.insert(0, '/kaggle/working/autosearch_kaggle')
from src.kaggle_secret import (load_api_key, SecretNotAttachedError,
                               SecretAuthError, SecretBackendError)
try:
    load_api_key('GEMINI_API_KEY')
except SecretNotAttachedError as exc:
    print('SECRET NOT ATTACHED:', exc)
    raise
except (SecretAuthError, SecretBackendError) as exc:
    print('SECRET FETCH FAILED:', type(exc).__name__, exc)
    raise
assert os.environ.get('GEMINI_API_KEY'), 'GEMINI_API_KEY secret missing or empty'
print('secret loaded:', bool(os.environ.get('GEMINI_API_KEY')))

In [ ]:
# 3. One-time data prep + tokenizer (small-GPU config from the T4 port).
import subprocess, sys
subprocess.run([sys.executable,'prepare.py','--num-shards','4'], cwd='repo', check=True)

In [ ]:
# 4. Phase 1 measurement (A8): one baseline run on GPU 0 through the real
# runner env (stripped_env forwards LD_LIBRARY_PATH so CUDA is visible).
# startup = total_seconds - training_seconds; set training.startup_allowance_s
# in config.yaml to ~2x that.
import subprocess, sys
from src.tools.runner import stripped_env
p = subprocess.run([sys.executable, 'train.py'], cwd='repo',
                   env=stripped_env(0),
                   capture_output=True, text=True, timeout=1500)
print('exit:', p.returncode)
print(p.stdout[-1500:]); print(p.stderr[-500:])
assert p.returncode == 0 and 'val_bpb:' in p.stdout, 'baseline run failed' 

In [ ]:
# 5. A7 check: two concurrent runs (GPU 0 + GPU 1) via stripped_env; compare
# spread vs the single run above. Full concurrent calibration = E13 (cell 6).
import subprocess, sys
from concurrent.futures import ThreadPoolExecutor
from src.tools.runner import stripped_env

def run_on(gpu):
    return subprocess.run([sys.executable, 'train.py'], cwd='repo',
                          env=stripped_env(gpu),
                          capture_output=True, text=True, timeout=1500)

with ThreadPoolExecutor(max_workers=2) as ex:
    procs = list(ex.map(run_on, [0, 1]))
for gpu, p in enumerate(procs):
    print(f'--- gpu{gpu} exit={p.returncode} ---')
    print(p.stdout[-400:]); print(p.stderr[-300:])
    assert p.returncode == 0 and 'val_bpb:' in p.stdout, f'gpu{gpu} run failed' 

In [ ]:
# 6. A/B evaluation (Section 9.1). Bounded SMOKE first (Phase 3/E14):
#   --config config.smoke.yaml --repeats 1 --arms multi_agent --out smoke_results
# Full campaign (Phase 6, only after approval + smoke green):
#   --config config.yaml --repeats 2 --out results
import subprocess, sys
p = subprocess.run([sys.executable, '-m', 'evals.run_ab',
                    '--config', 'config.smoke.yaml',
                    '--repeats', '1', '--arms', 'multi_agent',
                    '--out', 'smoke_results'],
                   text=True, capture_output=True)
print('exit:', p.returncode)
print(p.stdout[-3000:]); print(p.stderr[-1500:])
assert p.returncode == 0, 'run_ab failed — see stderr above' 

In [ ]:
# 7. Report + plots (smoke or full, whichever ran).
from pathlib import Path
for cand in ('smoke_results/ab_report.md', 'results/ab_report.md'):
    if Path(cand).exists():
        print('==', cand, '==')
        print(Path(cand).read_text())
try:
    from IPython.display import Image
    Image('results/val_bpb_curves.png')
except Exception as exc:
    print('plot not available:', exc)

In [ ]:
# 8. Persist. Resume later: attach this output as an input dataset, copy
# memory/ + work/ back into the project; orchestrator resumes from state.json.
import shutil
shutil.make_archive('/kaggle/working/autoresearch_results','zip','/kaggle/working')
print('saved /kaggle/working/autoresearch_results.zip')